In [1]:
import radiate as rd
import polars as pl
from IPython.display import display, HTML

rd.random.seed(67123)

In [2]:
def compute(x: float) -> float:
    return 4.0 * x**3 - 3.0 * x**2 + x


inputs = []
answers = []

input = -1.0
for _ in range(-10, 10):
    input += 0.1
    inputs.append([input])
    answers.append([compute(input)])

In [3]:
collector = rd.MetricCollector()

engine = (
    rd.Engine.graph(
        shape=(1, 1),
        vertex=[rd.Op.sub(), rd.Op.mul(), rd.Op.linear()],
        edge=rd.Op.weight(),
        output=rd.Op.linear(),
    )
    .regression(inputs, answers, loss=rd.MSE)
    .subscribe(collector)
    .alters(
        rd.Cross.graph(0.05, 0.5),
        rd.Mutate.op(0.07, 0.05),
        rd.Mutate.graph(0.1, 0.1, False),
    )
    .limit(rd.Limit.score(0.001), rd.Limit.generations(1000))
)

result = engine.run(log=True)

2026-06-03T18:19:40.971411Z  INFO Epoch 1    | Score:   2.0038 | Time: 175.04µs
2026-06-03T18:19:40.971501Z  INFO Epoch 2    | Score:   2.0038 | Time: 228.96µs
2026-06-03T18:19:40.971573Z  INFO Epoch 3    | Score:   2.0038 | Time: 277.42µs
2026-06-03T18:19:40.971635Z  INFO Epoch 4    | Score:   1.6821 | Time: 315.29µs
2026-06-03T18:19:40.971706Z  INFO Epoch 5    | Score:   1.6821 | Time: 360.50µs
2026-06-03T18:19:40.971784Z  INFO Epoch 6    | Score:   1.6821 | Time: 413.83µs
2026-06-03T18:19:40.971873Z  INFO Epoch 7    | Score:   1.6821 | Time: 478.00µs
2026-06-03T18:19:40.971955Z  INFO Epoch 8    | Score:   1.6821 | Time: 531.75µs
2026-06-03T18:19:40.972042Z  INFO Epoch 9    | Score:   1.6821 | Time: 587.33µs
2026-06-03T18:19:40.972125Z  INFO Epoch 10   | Score:   1.6821 | Time: 642.79µs
2026-06-03T18:19:40.972222Z  INFO Epoch 11   | Score:   1.6821 | Time: 710.33µs
2026-06-03T18:19:40.972309Z  INFO Epoch 12   | Score:   1.6821 | Time: 773.00µs
2026-06-03T18:19:40.972389Z  INFO Epoch 

In [4]:
# collector.plot()
print(result.metrics().dashboard())

[36 metrics, 50682 updates]
----- Metrics ----- (36 :: 50682) 
  carryover: 0.431  diversity: 0.571  unique_members: 66  unique_scores: 63  improvements: 114  iter_time(mean): 227.497µs

== Distributions ==
Name                       | Type   | Mean       | Min        | Max        | N      | Total        | StdDev     | Skew       | Kurt       | Entr      
-------------------------------------------------------------------------------------------------------------------------------------------------
age                        | dist   | 1.000      | 0.000      | 3.000      | 100    | 0.100        | 1.239      | 3.298      | 15.559     | -         
scores                     | dist   | 0.586      | 0.001      | 11.376     | 100    | 0.059        | 2.047      | 6.692      | 39.938     | -         
size.genome                | dist   | 26.110     | 26.000     | 27.000     | 100    | 2.611        | 0.314      | 0.000      | 0.000      | -         


== Statistics ==
Name                    

In [5]:
eval_results = result.value().eval(inputs)
accuracy = rd.accuracy(result.value(), inputs, answers, loss=rd.MSE)
accuracy

PyAccuracy("Regression Accuracy" {
	N: 20 
	Accuracy: 99.66%
	R² Score: 0.99976
	RMSE: 0.03118
	Loss (MSE): 0.00097
})

In [6]:
df = collector.to_polars(lazy=False)
df

name,last,sum,mean,stddev,var,skew,min,max,count,time_sum,time_mean,time_stddev,time_min,time_max,time_var,generation,update_count,tags
str,f64,f64,f64,f64,f64,f64,f64,f64,i64,duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],i64,i64,list[str]
"""count.evaluation""",8.0,108.0,54.0,65.053825,4232.0,NaN,8.0,100.0,2,null,null,null,null,null,null,0,2,"[""statistic""]"
"""step.evaluate.time""",0.000013,0.000073,0.000036,0.000034,1.1302e-9,NaN,0.000013,0.00006,2,72µs,36µs,33µs,12µs,60µs,0µs,0,2,"[""time"", ""step""]"
"""selector.roulette""",20.0,20.0,20.0,0.0,0.0,NaN,20.0,20.0,1,null,null,null,null,null,null,0,1,"[""selector"", ""statistic""]"
"""selector.roulette.time""",0.000005,0.000005,0.000005,0.0,0.0,NaN,0.000005,0.000005,1,5µs,5µs,0µs,5µs,5µs,0µs,0,1,"[""selector"", ""time""]"
"""selector.tournament""",80.0,80.0,80.0,0.0,0.0,NaN,80.0,80.0,1,null,null,null,null,null,null,0,1,"[""selector"", ""statistic""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""time""",0.000378,0.059604,0.000227,0.000197,3.8800e-8,0.0,0.000038,0.001826,262,59604µs,227µs,196µs,37µs,1826µs,0µs,261,1,"[""time""]"
"""index""",262.0,262.0,262.0,0.0,0.0,NaN,262.0,262.0,1,null,null,null,null,null,null,0,1,"[""statistic""]"
"""score.improvement""",1.0,114.0,1.0,0.0,0.0,0.0,1.0,1.0,114,null,null,null,null,null,null,261,1,"[""statistic"", ""score""]"


In [7]:
filtered = (
    df.filter(pl.col("name") == "score.improvement")
    .select("generation")
    .unique()
    .sort("generation")
)
filtered

generation
i64
0
3
96
102
103
…
255
257
258


In [8]:
display(HTML(filtered._repr_html_()))

generation
i64
0
3
96
102
103
…
255
257
258
